# MOD-06 · NB-05 — Instrumental Variables & Regression Discontinuity
### Health Analytics with Python · Module 06: Causal Inference in Health Research
---
**Learning objectives**
- Understand the three IV assumptions: relevance, exclusion restriction, independence
- Implement Two-Stage Least Squares (2SLS) from scratch
- Apply RDD with local linear regression and optimal bandwidth selection
- Test instrument strength with first-stage F-statistics
- Build sharp and fuzzy RDD designs for healthcare cutoffs
- Conduct sensitivity analyses (manipulation tests, bandwidth sensitivity)

**Estimated time:** 3 hours | **Level:** Advanced | **Libraries:** `statsmodels`, `numpy`, `scipy`


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from scipy.stats import norm
import warnings; warnings.filterwarnings("ignore")
import os; os.makedirs("/tmp/mod06", exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})
np.random.seed(42)


## 2. Instrumental Variables — Distance to Hospital

**Scenario:** Does receiving cardiac catheterisation cause lower mortality?
- Problem: sicker patients are more likely to receive the procedure (confounding)
- **Instrument:** Distance to nearest catheterisation-capable hospital
  - *Relevance:* Patients closer to cath-capable hospitals more likely to receive cath
  - *Exclusion:* Distance affects mortality ONLY through cath (not through other paths)
  - *Independence:* Distance is as-good-as-random (not correlated with patient severity)

**2SLS Estimator:**
1. First stage: Cath = alpha + gamma*Z + X*delta + v
2. Second stage: Mortality = beta0 + beta1*Cath_hat + X*pi + e


In [ ]:
# Simulate cardiac catheterisation data
np.random.seed(42); N = 3000
def sigmoid(x): return 1/(1+np.exp(-x))

# Instrument: distance to nearest cath hospital (miles)
distance = np.random.exponential(25, N).clip(1, 120)

# Unobserved severity (the confounder — related to treatment but not instrument)
severity = np.random.normal(0, 1, N)

# Treatment: cardiac catheterisation (influenced by both distance and severity)
cath_logit = 1.5 - 0.03*distance + 0.8*severity + np.random.normal(0, 0.5, N)
cath = np.random.binomial(1, sigmoid(cath_logit), N)

# Covariates
age     = np.random.normal(65, 12, N).clip(30, 90)
ami     = np.random.binomial(1, 0.6, N)  # acute MI

# Outcome: 30-day mortality
TRUE_CATH_EFFECT = -0.5  # log-odds cath reduces mortality
mortality_logit  = (-2.0 + TRUE_CATH_EFFECT*cath + 0.5*severity
                    + 0.02*(age-65) + 0.4*ami + np.random.normal(0, 0.4, N))
mortality = np.random.binomial(1, sigmoid(mortality_logit), N)

df_iv = pd.DataFrame({"distance":distance,"severity":severity,
                       "cath":cath,"age":age,"ami":ami,"mortality":mortality})
print(f"N={N} | Cath rate={cath.mean()*100:.1f}% | Mortality={mortality.mean()*100:.1f}%")
print(f"Cath rate by distance quartile:")
for q, (lo, hi) in enumerate(zip([0,25,40,60],[25,40,60,130])):
    sub = df_iv[(df_iv.distance>=lo)&(df_iv.distance<hi)]
    print(f"  Q{q+1} ({lo}-{hi} miles): {sub.cath.mean()*100:.1f}% cath | {sub.mortality.mean()*100:.1f}% mort")


In [ ]:
from statsmodels.sandbox.regression.gmm import IV2SLS

COVARIATES_IV = ["age","ami"]

# OLS (NAIVE — biased due to severity confounding)
X_ols = sm.add_constant(df_iv[["cath"] + COVARIATES_IV])
model_ols = sm.OLS(df_iv["mortality"], X_ols).fit()
ols_coef = model_ols.params["cath"]

# First stage: Cath ~ Distance + Covariates
X_fs = sm.add_constant(df_iv[["distance"] + COVARIATES_IV])
fs_model = sm.OLS(df_iv["cath"], X_fs).fit()
f_stat = fs_model.fvalue
cath_hat = fs_model.fittedvalues
print(f"First stage F-statistic: {f_stat:.1f}  ({'Strong' if f_stat>10 else 'Weak'} instrument, threshold = 10)")
print(f"First stage coefficient on distance: {fs_model.params['distance']:.4f} (should be negative)")

# Second stage: Mortality ~ Cath_hat + Covariates (manual 2SLS)
X_ss = sm.add_constant(pd.DataFrame({"cath_hat":cath_hat,
                                       "age":df_iv.age,"ami":df_iv.ami}))
model_2sls = sm.OLS(df_iv["mortality"], X_ss).fit()
iv_coef = model_2sls.params["cath_hat"]

print(f"\n{'Method':20s} {'Coef':>10s} {'OR':>10s} {'Bias vs True':>15s}")
print("-"*58)
for name, coef in [("OLS (naive)",ols_coef),("2SLS",iv_coef),("True",TRUE_CATH_EFFECT)]:
    bias = f"{coef-TRUE_CATH_EFFECT:+.3f}" if name!="True" else "—"
    print(f"  {name:18s} {coef:>10.4f} {np.exp(coef):>10.4f} {bias:>15s}")

# IV Visualisation
fig, axes = plt.subplots(1,2,figsize=(14,5))
# First stage
bins = np.percentile(distance, np.linspace(0,100,20))
dist_bins = pd.cut(distance, bins, labels=False)
bin_cath = df_iv.groupby(dist_bins)["cath"].mean()
bin_dist = [np.mean(distance[(dist_bins==b)]) for b in range(len(bin_cath))]
axes[0].scatter(bin_dist, bin_cath*100, s=40, color="#4878CF", zorder=3)
m,b = np.polyfit(bin_dist, bin_cath*100, 1)
xl = np.linspace(min(bin_dist),max(bin_dist),100)
axes[0].plot(xl, m*xl+b, "-", color="#D65F5F", lw=2)
axes[0].set_xlabel("Distance to cath hospital (miles)"); axes[0].set_ylabel("% receiving cath")
axes[0].set_title(f"First Stage: Distance -> Cath (F={f_stat:.0f})")

# Reduced form
bin_mort = df_iv.groupby(dist_bins)["mortality"].mean()
axes[1].scatter(bin_dist, bin_mort*100, s=40, color="#4878CF", zorder=3)
m2,b2 = np.polyfit(bin_dist, bin_mort*100, 1)
axes[1].plot(xl, m2*xl+b2, "-", color="#D65F5F", lw=2)
axes[1].set_xlabel("Distance to cath hospital (miles)"); axes[1].set_ylabel("30-day mortality (%)")
axes[1].set_title("Reduced Form: Distance -> Mortality")
plt.tight_layout(); plt.savefig("/tmp/mod06/iv_analysis.png",bbox_inches="tight"); plt.show()


## 3. Regression Discontinuity Design (RDD)

**Scenario:** Patients with LDL >= 190 mg/dL receive statin therapy per guidelines. Patients just below 190 do not.

The cutoff creates quasi-random assignment: patients just above and below 190 are similar on all characteristics, but differ in treatment probability.

**Sharp RDD:** Exact compliance at cutoff (everyone above gets treated)  
**Fuzzy RDD:** Probability of treatment jumps at cutoff (imperfect compliance)


In [ ]:
# Simulate LDL-based statin RDD
np.random.seed(42); N_rdd = 2000
CUTOFF = 190  # LDL mg/dL guideline threshold

ldl = np.random.normal(185, 20, N_rdd).clip(130, 250)
above_cutoff = (ldl >= CUTOFF).astype(int)
severity_rdd = np.random.normal(0, 1, N_rdd)

# SHARP RDD: treatment perfectly determined by cutoff
statin_sharp = above_cutoff.copy()

# FUZZY RDD: probability jumps at cutoff but not perfectly
def sigmoid(x): return 1/(1+np.exp(-x))
statin_prob = sigmoid(-2 + 5*above_cutoff + 0.5*np.random.normal(0,1,N_rdd))
statin_fuzzy = np.random.binomial(1, statin_prob, N_rdd)

# Outcome: cardiovascular event (5 years)
TRUE_RDD_EFFECT = -0.15  # risk difference
cv_prob = 0.25 + TRUE_RDD_EFFECT*statin_sharp - 0.002*(ldl-190) + 0.05*severity_rdd
cv_prob = cv_prob.clip(0.01, 0.99)
cv_event = np.random.binomial(1, cv_prob, N_rdd)

df_rdd = pd.DataFrame({"ldl":ldl,"above":above_cutoff,"statin_sharp":statin_sharp,
                        "statin_fuzzy":statin_fuzzy,"cv_event":cv_event,
                        "severity":severity_rdd})
df_rdd["running"] = df_rdd["ldl"] - CUTOFF  # centered at cutoff

print(f"RDD dataset: N={N_rdd}")
print(f"Above cutoff: {above_cutoff.mean()*100:.1f}%")
print(f"Statin (sharp): {statin_sharp.mean()*100:.1f}% | Statin (fuzzy): {statin_fuzzy.mean()*100:.1f}%")
print(f"CV event rate: {cv_event.mean()*100:.1f}%")


In [ ]:
# Sharp RDD estimation with local linear regression
BANDWIDTH = 15  # optimal bandwidth (in practice: use data-driven methods)

def rdd_estimate(df, running_col, outcome_col, treatment_col, bw, cutoff=0):
    in_bw = df[np.abs(df[running_col]) <= bw].copy()
    in_bw["D"] = (in_bw[running_col] >= 0).astype(int)
    in_bw["r_interact"] = in_bw[running_col] * in_bw["D"]
    model = smf.ols(f"{outcome_col} ~ {running_col} + D + r_interact",
                    data=in_bw).fit()
    tau = model.params["D"]
    ci  = model.conf_int().loc["D"]
    se  = model.bse["D"]
    return {"tau":tau,"se":se,"ci_lo":ci[0],"ci_hi":ci[1],"n":len(in_bw)}

rdd_est = rdd_estimate(df_rdd,"running","cv_event","statin_sharp",bw=BANDWIDTH)
print(f"Sharp RDD estimate (BW={BANDWIDTH}):")
print(f"  tau (RD) = {rdd_est['tau']:.4f}  (true: {TRUE_RDD_EFFECT:.4f})")
print(f"  SE       = {rdd_est['se']:.4f}")
print(f"  95% CI   : ({rdd_est['ci_lo']:.4f}, {rdd_est['ci_hi']:.4f})")
print(f"  N in BW  : {rdd_est['n']}")

# RDD plot
fig, axes = plt.subplots(1,2,figsize=(14,5))
bins = np.linspace(-30,30,25)
bin_labels = pd.cut(df_rdd["running"], bins, labels=False)
bin_mean_y = df_rdd.groupby(bin_labels)["cv_event"].mean()
bin_mean_x = [df_rdd.loc[bin_labels==b,"running"].mean() for b in range(len(bins)-1)]

axes[0].scatter(bin_mean_x, bin_mean_y*100, s=40, color="gray", zorder=3)
for side, color in [(1,"#D65F5F"),(-1,"#4878CF")]:
    mask = np.array(bin_mean_x)*side > 0
    m,b_fit = np.polyfit(np.array(bin_mean_x)[mask], np.array(list(bin_mean_y.values))[mask]*100, 1)
    xl = np.linspace(0,30*side,50) if side>0 else np.linspace(-30,0,50)
    axes[0].plot(xl, m*xl+b_fit, "-", color=color, lw=2)
axes[0].axvline(0, color="black", ls="--", lw=1.5, label="Cutoff (LDL=190)")
axes[0].set_xlabel("LDL - 190 (mg/dL)"); axes[0].set_ylabel("5-year CV event rate (%)")
axes[0].set_title("Sharp RDD: Jump in outcome at LDL=190")
axes[0].legend(fontsize=9)

# Bandwidth sensitivity
bws = range(5, 35, 2)
bw_results = [rdd_estimate(df_rdd,"running","cv_event","statin_sharp",bw=b) for b in bws]
taus = [r["tau"] for r in bw_results]; ses = [r["se"] for r in bw_results]
axes[1].plot(bws, taus, "o-", color="#4878CF", lw=2, ms=6)
axes[1].fill_between(bws, np.array(taus)-1.96*np.array(ses),
                         np.array(taus)+1.96*np.array(ses), alpha=0.2, color="#4878CF")
axes[1].axhline(TRUE_RDD_EFFECT, color="#D65F5F", ls="--", lw=2, label=f"True effect ({TRUE_RDD_EFFECT})")
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_xlabel("Bandwidth (LDL mg/dL from cutoff)"); axes[1].set_ylabel("RDD estimate (tau)")
axes[1].set_title("Bandwidth Sensitivity Analysis"); axes[1].legend(fontsize=9)
plt.tight_layout(); plt.savefig("/tmp/mod06/rdd_analysis.png",bbox_inches="tight"); plt.show()


## 4. Knowledge Check
1. First-stage F-statistic = 8.2. Is this instrument strong enough? What are the implications for your 2SLS estimates?
2. The RDD estimate changes from -0.18 to -0.09 as bandwidth increases from 5 to 25. What does this pattern suggest?
3. Patients might manipulate their LDL measurement to stay below 190 to avoid statins. How would you test for this?
4. What does the LATE (Local Average Treatment Effect) represent in the cardiac catheterisation IV analysis?
5. Implement a density test (McCrary test) to check for manipulation at the LDL cutoff.


In [ ]:
# Exercise 5 — McCrary density test (simplified)
# Test: Is there an unusual density jump at the cutoff?
# If patients manipulate, we'd see excess density just below 190

from scipy.stats import gaussian_kde
bw_kde = 3.0
kde = gaussian_kde(df_rdd["ldl"], bw_method=bw_kde/df_rdd["ldl"].std())
ldl_grid = np.linspace(150, 230, 200)
density = kde(ldl_grid)

fig, ax = plt.subplots(figsize=(9,4))
ax.plot(ldl_grid, density, "-", color="#4878CF", lw=2)
ax.axvline(CUTOFF, color="#D65F5F", ls="--", lw=2, label=f"Cutoff (LDL={CUTOFF})")
ax.set_xlabel("LDL (mg/dL)"); ax.set_ylabel("Density")
ax.set_title("McCrary Density Test: Check for Manipulation at Cutoff")
ax.legend()
plt.tight_layout(); plt.show()

# Local density comparison just left and right of cutoff
eps = 8  # window around cutoff
left_density  = ((df_rdd["ldl"] >= CUTOFF-eps) & (df_rdd["ldl"] < CUTOFF)).mean()
right_density = ((df_rdd["ldl"] >= CUTOFF) & (df_rdd["ldl"] < CUTOFF+eps)).mean()
density_ratio = right_density/left_density
print(f"Density just left of cutoff  ({CUTOFF-eps}-{CUTOFF}): {left_density:.4f}")
print(f"Density just right of cutoff ({CUTOFF}-{CUTOFF+eps}): {right_density:.4f}")
print(f"Density ratio (right/left): {density_ratio:.3f}  (should be ~1.0 if no manipulation)")
print(f"Interpretation: {'No evidence of manipulation' if 0.8<density_ratio<1.25 else 'Potential manipulation!'}")


---
**Next -> NB-06: G-Computation & TMLE**